# 22 · Reranker：召回之后的精排

> 向量检索“快而粗”（Bi-Encoder，各自编码），Reranker“慢而准”（Cross-Encoder，成对编码）。两段式是生产 RAG 的标准架构。

**本文件覆盖知识点**：两段式流程(Retriever→Reranker→LLM) / Cross Encoder / Bi-Encoder / Late Interaction / LLM Reranker / BGE·Jina·Cohere·BGE-v2

```text
Query → Retriever → Top 50 → Reranker → Top 5 → LLM
        (粗召回)      (候选池)   (精排)     (进上下文)
```

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Bi-Encoder vs Cross-Encoder

| | Bi-Encoder | Cross-Encoder |
|--|-----------|---------------|
| 编码 | query、doc **各自**编码成向量 | query 与每个 doc **拼接后**一起编码 |
| 交互 | 向量点积，无深层交互 | 逐 token 深度交互 |
| 精度 | 粗 | 更精细 |
| 速度 | 快（向量可预计算、建索引） | 慢（每条候选都要过一次模型） |
| 用于 | **召回**（Retriever） | **精排**（Reranker） |

> **为什么 bge-reranker 叫 Cross-Encoder？** 因为它把 (query, doc) 拼成一个输入喂给 Transformer，让两边的 token 互相“看见”，从而判断细粒度相关性——代价是无法预计算 doc 向量，只能在线逐条打分，故只能用于少量候选的精排。

In [ ]:
# .env 配置
from dotenv import load_dotenv; load_dotenv()
import os
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

# 用百炼 gte-rerank 做精排（DashScope 的 TextReRank 接口）
def rerank(query, documents, top_n=3):
    from dashscope import TextReRank
    if not documents:
        return []
    r = TextReRank.call(model='gte-rerank', query=query, documents=documents,
                        top_n=min(top_n, len(documents)), api_key=API_KEY)
    if r.status_code != 200:
        print('重排失败，退回原顺序:', r.message)
        return [(d, 0.0) for d in documents[:top_n]]
    return [(documents[it['index']], it['relevance_score']) for it in r.output['results']]

query = '星云客服机器人如何计费？'
cands = [
    '提供免费试用额度，付费套餐分基础版、专业版、企业版三档',
    '支持公有云 SaaS 与私有化两种部署方式',
    'RAG 通过检索外部知识库增强回答',
]
if API_KEY and '你的' not in API_KEY:
    for text, score in rerank(query, cands):
        print(f'  {score:+.3f}  {text[:24]}')
else:
    print('gte-rerank 会把最“扣题”的计费片段排到最前（配置 .env 后运行）。')

## 2. 主流 Reranker

| 模型 | 类型 | 备注 |
|------|------|------|
| **BGE-reranker / v2 / v2-m3** | Cross | 中文强，v2-m3 多语言 |
| **Cohere Rerank** | 服务 API | 上手简单、多语 |
| **Jina Reranker** | Cross | 8K 长候选 |
| **gte-rerank(百炼)** | Cross 服务 | 本课程使用 |

另外两类：
- **Late Interaction（如 ColBERT）**：doc 仍可预编码，但保留每 token 向量，在线做 MaxSim——召回级也能享受“交互”红利（第 39 课）；
- **LLM Reranker**：直接让大模型对候选排序/打分，灵活但贵，适合小候选集。

In [ ]:
# 知识点·真调说明：LLM Reranker —— 让大模型亲自当“精排器”，把候选按相关性重新排序并剔除噪声
import json as _json
rerank_query = '星云客服机器人标准版支持私有化部署吗？'
rerank_cands = [
    '标准版默认走公有云 SaaS，如需私有化部署需联系销售单独开通。',   # 编号1：正面命中
    '支持免费试用额度，付费套餐分基础版、标准版、专业版三档。',       # 编号2：沾边
    '竞品天穹客服主打私有化，最低 20 万一年。',                        # 编号3：貌似相关实则跑题
    '部署方式支持公有云、专有云与本地化，详见部署指南第三章。',       # 编号4：正面命中
    '标准版 998 元/月，含 5 个坐席与基础报表。',                       # 编号5：沾边
]
print('① 检索召回（原顺序，未经精排）:')
for i, t in enumerate(rerank_cands, 1):
    print('  [%s] %s' % (i, t))
print()
cand_list = '\n'.join('[%s] %s' % (i + 1, t) for i, t in enumerate(rerank_cands))
out = _llm_live(
    prompt='问题：%s\n\n候选片段：\n%s\n\n请扮演精排重排器，把候选按与问题的相关度从高到低排序。' % (rerank_query, cand_list),
    system='你是 RAG 的 LLM Reranker（精排器）。规则：针对给定问题，把候选片段按相关度从高到低重排；'
           '明显不相关的直接剔除、不要排在后面充数。只输出一个 JSON，禁止任何其它文字：'
           '{"ordered_ids": [按相关度从高到低的候选编号数组], "reason": "一句话解释为何这样排"}。',
    fallback='未配置 Key 的固定样例：\n'
             '{"ordered_ids": [1, 4], "reason": "候选1和4正面回答标准版/私有化部署，其余只是沾边或讲竞品。"}',
    temperature=0.1,
)
if out is None:
    out = '{"ordered_ids": [1, 4], "reason": "候选1和4正面回答标准版/私有化部署，其余只是沾边或讲竞品。"}'
    print('（以上为固定样例；下面用样例演示 JSON 解析）')
try:
    seg = out[out.find('{'): out.rfind('}') + 1]
    obj = _json.loads(seg)
    ids = obj['ordered_ids']
    print('② LLM Reranker 精排后（越靠前越相关）:')
    for k, cid in enumerate(ids, 1):
        print('  %d. [%s] %s' % (k, cid, rerank_cands[int(cid) - 1][:28]))
    dropped = [c for c in range(1, len(rerank_cands) + 1) if c not in ids]
    print('  被剔除（判为不相关）:', dropped or '无')
    print('  排序理由:', obj.get('reason', ''))
except Exception as e:
    print('JSON 解析失败：', e, '—— 说明对 LLM Reranker 的输出约束还要再收紧。')
print('→ LLM 能“读懂语义”来精排，但每条候选都要过一次模型、慢而贵，所以只能作用于小候选集；'
      '这就是它和可预计算向量的 Cross-Encoder 精排（上面 gte-rerank）的取舍。')

## 3. 工程要点

- **候选池大小**：召回 Top-50~100 → 精排只留 Top-3~5（池太浅会漏正确项）；
- **失败降级**：Reranker 挂了要能退回召回顺序（代码里已示范）；
- **缓存打分**：同一 (query, doc) 对缓存结果，减少重复计算；
- **验收**：对比“召回后直接 Top-K”与“再重排”的答案质量/命中率。

## 小结

- 两段式：**Bi-Encoder 召回 + Cross-Encoder 精排**；
- Reranker 精但慢，只作用于小候选池；
- 进阶有 Late Interaction / LLM Rerank。